# CNN trên FashionMNIST: vì sao convolution thắng MLP trên ảnh

FashionMNIST gồm 70 000 ảnh xám 28×28 của 10 loại quần áo, giày, túi (Zalando Research): 60 000
ảnh train, 10 000 ảnh test, mỗi lớp đúng 10%. Bạn sẽ dựng một MLP và một CNN nhỏ, train cả hai trên
CPU, và đọc confusion matrix để thấy model sai ở đâu.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) tải dữ liệu thật và train model; test bỏ qua. Lần
  đầu chạy sẽ tải khoảng 30 MB vào `data/raw/fashion-mnist`.
- Test dùng ảnh giả 8×8 sinh trong test, nên mọi hàm phải chạy với **kích thước ảnh bất kỳ** chia
  hết cho 4, không chỉ 28.
- Chấm: `uv run pytest tracks/data-science/modules/13-deep-learning/exercises/intermediate/01-fashion-cnn -q`

In [ ]:
import time

import torch
from torch import nn

## 1. Chuẩn hoá

`standardize(images, mean=None, std=None)` nhận tensor `uint8` shape `(N, H, W)`:

- chia 255 về `[0, 1]`, rồi trừ `mean`, chia `std` (một số cho cả ảnh, vì ảnh xám);
- `mean`, `std` là `None` thì tính từ chính `images` (chỉ làm vậy với tập **train**); có giá trị thì
  dùng lại (cho validation, test);
- thêm chiều kênh: trả `(X float32 (N, 1, H, W), mean: float, std: float)`.

In [ ]:
def standardize(images: torch.Tensor, mean: float | None = None, std: float | None = None):
    """uint8 (N, H, W) → (float32 (N, 1, H, W) đã chuẩn hoá, mean, std)."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: tải FashionMNIST (một lần), tách validation từ train, chuẩn hoá bằng thống kê của train
from pathlib import Path

from torchvision.datasets import FashionMNIST

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
data_dir = ROOT / "data" / "raw" / "fashion-mnist"
train_raw = FashionMNIST(data_dir, train=True, download=True)
test_raw = FashionMNIST(data_dir, train=False, download=True)
CLASSES = train_raw.classes

X_all, MEAN, STD = standardize(train_raw.data[:50000])
X_train, y_train = X_all, train_raw.targets[:50000]
X_val, _, _ = standardize(train_raw.data[50000:], MEAN, STD)
y_val = train_raw.targets[50000:]
X_test, _, _ = standardize(test_raw.data, MEAN, STD)
y_test = test_raw.targets
print(X_train.shape, X_val.shape, X_test.shape, round(MEAN, 4), round(STD, 4))
print(dict(zip(CLASSES, y_test.bincount().tolist())))

## 2. Hai kiến trúc

`build_mlp` đã viết sẵn làm mẫu. Viết `build_cnn(image_size, n_classes, channels=(32, 64), dropout=0.3)`:

- hai khối `Conv2d(kernel 3, padding 1) → ReLU → MaxPool2d(2)`, số kênh lần lượt `channels`;
- sau hai lần pool, ảnh còn `image_size // 4` mỗi chiều: `Flatten → Dropout → Linear(…, 128) → ReLU
  → Dropout → Linear(128, n_classes)`;
- `image_size` không chia hết cho 4 thì `raise ValueError`.

`count_params(model)` đếm số tham số **train được** (`requires_grad`).

In [ ]:
def build_mlp(image_size: int, n_classes: int, hidden=(256, 128), dropout: float = 0.2) -> nn.Module:
    layers, width = [nn.Flatten()], image_size * image_size
    for h in hidden:
        layers += [nn.Linear(width, h), nn.ReLU(), nn.Dropout(dropout)]
        width = h
    layers.append(nn.Linear(width, n_classes))
    return nn.Sequential(*layers)


def build_cnn(image_size: int, n_classes: int, channels=(32, 64), dropout: float = 0.3) -> nn.Module:
    """Hai khối conv-relu-pool rồi hai tầng Linear; output là logits (N, n_classes)."""
    # TODO
    raise NotImplementedError


def count_params(model: nn.Module) -> int:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: số tham số từng tầng. Phần lớn tham số của CNN nằm ở tầng Linear đầu tiên, không phải ở conv.
for name, model in [("MLP", build_mlp(28, 10)), ("CNN", build_cnn(28, 10))]:
    print(name, count_params(model))
    for layer in model:
        n = count_params(layer)
        if n:
            print("   ", layer, n)

## 3. Train, dự đoán, confusion matrix

- `train_model(model, X, y, epochs, lr=1e-3, batch_size=128, seed=0) -> list[float]`: Adam,
  `CrossEntropyLoss`, mỗi epoch xáo thứ tự bằng `torch.Generator().manual_seed(seed + epoch)`,
  trả loss trung bình theo mẫu của từng epoch. Cùng seed và cùng trọng số khởi tạo → cùng kết quả.
- `predict(model, X, batch_size=1000) -> LongTensor (N,)`: chế độ eval, không gradient, theo batch
  (10 000 ảnh một lượt vẫn chạy được, nhưng ảnh lớn hơn thì hết RAM).
- `confusion_matrix(y_true, y_pred, n_classes)`: hàng là nhãn thật, cột là dự đoán.

In [ ]:
def train_model(model: nn.Module, X: torch.Tensor, y: torch.Tensor, epochs: int, lr: float = 1e-3,
                batch_size: int = 128, seed: int = 0) -> list[float]:
    # TODO: giống train_one_epoch của bài Basic 02, nhưng tự chia batch bằng torch.randperm
    raise NotImplementedError


def predict(model: nn.Module, X: torch.Tensor, batch_size: int = 1000) -> torch.Tensor:
    # TODO
    raise NotImplementedError


def confusion_matrix(y_true: torch.Tensor, y_pred: torch.Tensor, n_classes: int) -> torch.Tensor:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: train MLP và CNN cùng 5 epoch trên 50 000 ảnh (CPU). In thời gian và accuracy validation/test.
print("CPU threads:", torch.get_num_threads())
results = {}
for name, build in [("MLP", build_mlp), ("CNN", build_cnn)]:
    torch.manual_seed(0)
    model = build(28, 10)
    start = time.perf_counter()
    losses = train_model(model, X_train, y_train, epochs=5, lr=1e-3, batch_size=128, seed=0)
    seconds = time.perf_counter() - start
    val_acc = (predict(model, X_val) == y_val).float().mean().item()
    test_acc = (predict(model, X_test) == y_test).float().mean().item()
    results[name] = model
    print(f"{name}: {count_params(model):,} tham số, {seconds:.0f} s, loss {[round(v, 3) for v in losses]}, "
          f"val {val_acc:.4f}, test {test_acc:.4f}")

In [ ]:
# explore: CNN nhầm lớp nào với lớp nào? In các cặp nhầm nhiều nhất và accuracy từng lớp.
cnn = results["CNN"]
cm = confusion_matrix(y_test, predict(cnn, X_test), 10)
per_class = (cm.diag() / cm.sum(1)).tolist()
for c, acc in sorted(zip(CLASSES, per_class), key=lambda t: t[1]):
    print(f"{c:12s} {acc:.3f}")
off = cm.clone().fill_diagonal_(0)
top = off.flatten().topk(5)
for count, flat in zip(top.values.tolist(), top.indices.tolist()):
    print(f"thật {CLASSES[flat // 10]:12s} → đoán {CLASSES[flat % 10]:12s} {count}")

In [ ]:
# explore: xem vài ảnh Shirt bị đoán nhầm
import matplotlib.pyplot as plt

pred = predict(cnn, X_test)
wrong = ((y_test == 6) & (pred != 6)).nonzero().flatten()[:8]
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, i in zip(axes, wrong.tolist()):
    ax.imshow(test_raw.data[i], cmap="gray")
    ax.set_title(CLASSES[pred[i]], fontsize=8)
    ax.axis("off")
plt.show()

## 4. Câu hỏi (viết câu trả lời vào cell này)

1. CNN có nhiều tham số hơn MLP, vậy vì sao nói convolution "tiết kiệm tham số"? Tầng nào chiếm
   phần lớn tham số của CNN, và bạn sẽ đổi gì để giảm mạnh con số đó?
2. Lớp nào khó nhất và nó bị nhầm với lớp nào? Một người nhìn ảnh có làm tốt hơn nhiều không?
3. Model này train 5 epoch cố định. Nếu muốn chọn số epoch, bạn dùng tập nào, và vì sao không dùng
   tập test?